In [1]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

In [2]:
so_bugs = pd.read_excel("Dataset/so_bugs.xlsx")
gh_bugs = pd.read_excel("Dataset/gh_bugs.xlsx")

print(f"Stack Overflow: {len(so_bugs)} bugs")
print(f"GitHub        : {len(gh_bugs)} bugs")

Stack Overflow: 1004 bugs
GitHub        : 180 bugs


In [3]:
columns_to_analyze = [
    "Symptom Category",
    "Symptom",
    "Root Cause Category",
    "Root Cause",
    "Fix Pattern Category",
    "Fix Pattern",
]

In [4]:
def perform_mannwhitney_test(df1, df2, column):
    # Turn string values to numbers
    data1_raw = df1[column].dropna()
    data2_raw = df2[column].dropna()
    all_data = pd.concat([data1_raw, data2_raw])
    freq_order = {value: idx for idx, value in enumerate(all_data.value_counts().index)}
    data1 = np.array([freq_order[x] for x in data1_raw])
    data2 = np.array([freq_order[x] for x in data2_raw])

    statistic, p_value = mannwhitneyu(data1, data2)

    # calculate effect size
    n1, n2 = len(data1), len(data2)
    n = n1 + n2
    z_score = (statistic - (n1 * n2 / 2)) / np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
    effect_size = abs(z_score) / np.sqrt(n)

    return p_value, effect_size

In [5]:
def holm_bonferroni_correction(p_values):
    n = len(p_values)
    indexed_p = [(i, p) for i, p in enumerate(p_values)]
    indexed_p.sort(key=lambda x: x[1])

    corrected_p = [0.0] * n
    for idx, (original_idx, p) in enumerate(indexed_p):
        rank = idx + 1
        corrected_p[original_idx] = min(p * (n - rank + 1), 1.0)

    return np.array(corrected_p)

In [6]:
results = []
p_values = []

for column in columns_to_analyze:
    p_value, effect_size = perform_mannwhitney_test(so_bugs, gh_bugs, column)
    results.append([column, effect_size])
    p_values.append(p_value)

results = pd.DataFrame(results, columns=["Template Engine", "Effect Size"])
results["p-values"] = holm_bonferroni_correction(p_values)

pd.set_option("display.float_format", lambda x: "%.6f" % x)
results[["Template Engine", "p-values", "Effect Size"]].sort_values(
    "p-values", ascending=False
)

,Template Engine,p-values,Effect Size
1,Symptom,0.538922,0.017615
2,Root Cause Category,0.159064,0.049556
0,Symptom Category,0.132046,0.054592
5,Fix Pattern,0.009229,0.087749
4,Fix Pattern Category,0.000115,0.104402
3,Root Cause,0.000000,0.174431


In [7]:
def bug_stats(df):
    results = {}
    for column in columns_to_analyze:
        data = df[column].dropna()
        value_counts = data.value_counts()
        total_count = len(data)

        stats = []
        for value, count in value_counts.items():
            percentage = (count / total_count) * 100
            stats.append({"value": value, "count": count, "percentage": percentage})
        results[column] = stats

    return results

so_stats = bug_stats(so_bugs)
gh_stats = bug_stats(gh_bugs)

In [8]:
def compare(column_name, so_stats, gh_stats):
    so_stats_dict = {stat["value"]: stat for stat in so_stats.get(column_name)}
    gh_stats_dict = {stat["value"]: stat for stat in gh_stats.get(column_name)}

    sorted_values = []
    for stat in so_stats_dict:
        sorted_values.append(stat)

    data = []

    for value in sorted_values:
        so_stat = so_stats_dict.get(value)
        gh_stat = gh_stats_dict.get(value)

        if so_stat:
            so_text = f"{so_stat['count']} / {so_stat['percentage']:.2f}%"

        if gh_stat:
            gh_text = f"{gh_stat['count']} / {gh_stat['percentage']:.2f}%"
        else:
            gh_text = "0 / 0.00%"

        data.append([value, so_text, gh_text])

    return pd.DataFrame(data, columns=["Dimension", "Stack Overflow", "GitHub"])

In [9]:
compare(columns_to_analyze[0], so_stats, gh_stats)

,Dimension,Stack Overflow,GitHub
0,Abnormal Rendering Result,488 / 48.61%,83 / 46.11%
1,Compilation Error,238 / 23.71%,31 / 17.22%
2,Placeholder Error,181 / 18.03%,25 / 13.89%
3,Initialization Error,77 / 7.67%,40 / 22.22%
4,Others,20 / 1.99%,1 / 0.56%


In [10]:
compare(columns_to_analyze[1], so_stats, gh_stats)

,Dimension,Stack Overflow,GitHub
0,Unexpected Output,267 / 26.59%,63 / 35.00%
1,Invalid Expression,153 / 15.24%,16 / 8.89%
2,Blank Output,113 / 11.25%,14 / 7.78%
3,Undefined Variable,78 / 7.77%,13 / 7.22%
4,Resource Not Found,76 / 7.57%,3 / 1.67%
5,Template Not Found,60 / 5.98%,23 / 12.78%
6,Property Access Error,55 / 5.48%,10 / 5.56%
7,Type Mismatch,48 / 4.78%,2 / 1.11%
8,Bad Delimiter,47 / 4.68%,4 / 2.22%
9,Unrecognized Control Structure,38 / 3.78%,11 / 6.11%


In [11]:
compare(columns_to_analyze[2], so_stats, gh_stats)

,Dimension,Stack Overflow,GitHub
0,Syntax Misuse,359 / 35.76%,38 / 21.11%
1,Mismatched Data Context,195 / 19.42%,31 / 17.22%
2,Incompatible Integration,168 / 16.73%,61 / 33.89%
3,Incorrect Property Resolution,109 / 10.86%,13 / 7.22%
4,Improper Configuration,92 / 9.16%,16 / 8.89%
5,Mechanism Misconception,81 / 8.07%,21 / 11.67%


In [12]:
compare(columns_to_analyze[3], so_stats, gh_stats)

,Dimension,Stack Overflow,GitHub
0,Expression Misuse,197 / 19.62%,14 / 7.78%
1,Inconsistent Data Type,139 / 13.84%,3 / 1.67%
2,Control Structure Misuse,108 / 10.76%,16 / 8.89%
3,Invalid Access Semantic,72 / 7.17%,2 / 1.11%
4,Incorrect Resource Reference,67 / 6.67%,3 / 1.67%
5,Template File Path Misconfiguration,59 / 5.88%,24 / 13.33%
6,Incomplete Data Context,56 / 5.58%,28 / 15.56%
7,Delimiter Misuse,54 / 5.38%,8 / 4.44%
8,Language Convention,45 / 4.48%,0 / 0.00%
9,Incorrect Request,38 / 3.78%,9 / 5.00%


In [13]:
compare(columns_to_analyze[4], so_stats, gh_stats)

,Dimension,Stack Overflow,GitHub
0,Template-side Fix,667 / 67.92%,94 / 52.22%
1,Host-side Fix,203 / 20.67%,50 / 27.78%
2,Configuration-level Fix,112 / 11.41%,36 / 20.00%


In [14]:
compare(columns_to_analyze[5], so_stats, gh_stats)

,Dimension,Stack Overflow,GitHub
0,Template Logic Modification,315 / 32.08%,35 / 19.44%
1,Template Syntax Correction,173 / 17.62%,34 / 18.89%
2,Data Context Refinement,138 / 14.05%,12 / 6.67%
3,Property Resolution Correction,66 / 6.72%,16 / 8.89%
4,Application Logic Refactoring,65 / 6.62%,38 / 21.11%
5,Resource Path Correction,61 / 6.21%,2 / 1.11%
6,Template Path Alignment,51 / 5.19%,23 / 12.78%
7,Engine Property Tuning,42 / 4.28%,8 / 4.44%
8,Template Logic Offloading,30 / 3.05%,0 / 0.00%
9,Validity Check,22 / 2.24%,7 / 3.89%
